**SENTIMENT ANALYSIS ON IMDB DATASET**

In [27]:
!pip install -q streamlit

In [28]:
%%writefile app.py

import streamlit as st
import numpy as np
import re

from tensorflow.keras.models import load_model
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_features = 10000
max_len = 500

@st.cache_resource
def load_sentiment_model():
    return load_model("gru_sentiment_model.keras")

model = load_sentiment_model()

@st.cache_data
def get_word_index():
    return imdb.get_word_index()

word_index = get_word_index()

def predict_sentiment(review):

    review = review.lower()
    review = re.sub(r"[^\w\s']", " ", review)

    words = review.split()

    encoded = [1]

    for word in words:
        index = word_index.get(word)

        if index is not None and index + 3 < max_features:
            encoded.append(index + 3)
        else:
            encoded.append(2)

    padded = pad_sequences(
        [encoded],
        maxlen=max_len
    )

    prediction = model.predict(padded, verbose=0)[0][0]

    return prediction


st.title("IMDB Movie Sentiment Analysis")

st.write("Sentiment Prediction using GRU")

review = st.text_area("Enter Movie Review")

if st.button("Predict"):

    if review.strip() == "":
        st.warning("Please enter a review")

    else:
        prediction = predict_sentiment(review)

        if prediction >= 0.5:
            st.success("Positive Review 😊")
        else:
            st.error("Negative Review")

        st.write("Positive Probability:", float(prediction))

Writing app.py


In [29]:
!streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [30]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!./cloudflared tunnel --url http://localhost:8501

2026-11-09T11:54:39Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-09T11:54:39Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-09T11:54:42Z INF +--------------------------------------------------------------------------------------------+
2026-11-09T11:54:42Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-09T11:54:42Z INF |  https://recorded-lafayette-competent-silly.trycloudfl